# 4. Xây Dựng Mô Hình Machine Learning - Dự Đoán Churn

Notebook này trình bày quá trình xây dựng mô hình ML để dự đoán khách hàng rời bỏ (Churn Prediction)

**Bài toán**: Cho một khách hàng với các đặc trưng hành vi (tần suất mua, giá trị chi tiêu, đánh giá, thời gian giao hàng...), dự đoán xem khách đó có rời bỏ hay không.

**Loại bài toán**: Binary Classification (Phân loại nhị phân)
- Lớp 0: Khách hàng KHÔNG rời bỏ (vẫn hoạt động)
- Lớp 1: Khách hàng ĐÃ rời bỏ (churn)

## 4.1 Vấn đề Data Leakage - Tại sao KHÔNG dùng Recency làm feature?

### Data Leakage là gì?
Data leakage xảy ra khi **thông tin từ biến mục tiêu (target)** bị rò rỉ vào features, khiến mô hình "nhìn thấy đáp án" trong lúc huấn luyện.

### Vấn đề cụ thể trong bài toán này:
- **Target**: `churn = 1` khi `recency > 90` ngày
- **Feature bị rò rỉ**: `recency` (số ngày từ lần mua cuối)

Nếu dùng `recency` làm feature, mô hình chỉ cần kiểm tra `recency > 90` là biết ngay `churn = 1`, cho ra kết quả **Accuracy = 1.0000, AUC = 1.0000** — trông "hoàn hảo" nhưng **hoàn toàn vô nghĩa** vì model không học được gì thực sự.

### Giải pháp:
Loại bỏ `recency`, `r_score` khỏi danh sách features. Chỉ giữ lại các đặc trưng **hành vi** thực sự

## 4.2 Import thư viện và khởi tạo Spark

In [17]:
import os
os.environ["PYSPARK_PYTHON"] = "C:/Users/Admin/AppData/Local/Programs/Python/Python313/python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = os.environ["PYSPARK_PYTHON"]
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["TZ"] = "UTC"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import RandomForestClassifier, LogisticRegression
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator,
)
from pyspark.ml.functions import vector_to_array

spark = (
    SparkSession.builder
    .appName("Olist_ML_ChurnPrediction")
    .master("local[4]")
    .config("spark.driver.memory", "4g")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.ui.enabled", "false")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.network.timeout", "800s")
    .config("spark.sql.session.timeZone", "UTC")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("SparkSession đã khởi tạo thành công!")
print(f"   Spark version: {spark.version}")

SparkSession đã khởi tạo thành công!
   Spark version: 3.5.1


## 4.3 Đọc dữ liệu từ HDFS

Đọc 2 nguồn dữ liệu:
- **`merged_orders`** (Silver): Chứa thông tin đơn hàng chi tiết → dùng để tính thêm features
- **`rfm_customers`** (Gold): Chứa RFM per customer → dùng để tạo label `churn`

In [18]:
HDFS_SILVER = "hdfs://localhost:9000/user/bigdata/olist/silver"
HDFS_GOLD = "hdfs://localhost:9000/user/bigdata/olist/gold"

# Đọc merged_orders
merged_df = spark.read.parquet(f"{HDFS_SILVER}/merged_orders")
print(f"merged_orders: {merged_df.count():,} dòng, {len(merged_df.columns)} cột")

# Đọc rfm_customers
rfm_df = spark.read.parquet(f"{HDFS_GOLD}/rfm_customers")
print(f"rfm_customers: {rfm_df.count():,} dòng, {len(rfm_df.columns)} cột")

merged_orders: 99,277 dòng, 50 cột
rfm_customers: 93,358 dòng, 12 cột


## 4.4 Chuẩn bị Features cho Mô hình

### Chiến lược Feature Engineering cho Churn Prediction:

**1**: Từ `merged_orders`, tính thêm các đặc trưng **hành vi** per customer:
- `avg_review`: Đánh giá trung bình → phản ánh mức hài lòng
- `avg_delivery_days`: Thời gian giao hàng TB → trải nghiệm dịch vụ
- `total_items_bought`: Tổng SP đã mua → mức độ gắn kết
- `avg_payment`: Giá trị thanh toán TB → sức mua
- `category_diversity`: Số danh mục đã mua → mức độ khám phá

**2**: Từ `rfm_customers`, lấy `monetary`, `m_score` (KHÔNG lấy `recency`, `r_score`!)

**3**: Tạo label `churn` từ `recency` (churn=1 nếu recency > 90 ngày)

In [19]:
# === BƯỚC 1: Tính thêm features per customer từ merged_orders ===
print("Bước 1: Tính features hành vi per customer...")
customer_features = (
    merged_df
    .groupBy("customer_unique_id")
    .agg(
        F.avg("review_score").alias("avg_review"),           # Đánh giá TB
        F.avg("delivery_days").alias("avg_delivery_days"),        # Ngày giao hàng TB
        F.sum("total_items").alias("total_items_bought"),         # Tổng SP đã mua
        F.avg("total_payment_value").alias("avg_payment"),        # Giá trị thanh toán TB
        F.countDistinct("main_category_english").alias("category_diversity"),  # Số danh mục
    )
)
print(f"   → customer_features: {customer_features.count():,} khách hàng")
customer_features.show(5, truncate=False)

Bước 1: Tính features hành vi per customer...
   → customer_features: 95,991 khách hàng
+--------------------------------+----------+-----------------+------------------+-----------+------------------+
|customer_unique_id              |avg_review|avg_delivery_days|total_items_bought|avg_payment|category_diversity|
+--------------------------------+----------+-----------------+------------------+-----------+------------------+
|aaeafdd832692cf3b77f4eeded665088|5.0       |21.0             |1                 |92.74      |1                 |
|7113a50db7ae2bc6f93d766749e5125a|5.0       |8.0              |1                 |48.01      |1                 |
|e4c5964975cdce8052a5743336264234|5.0       |9.0              |1                 |32.69      |1                 |
|0242e42b2210c764b47e099ddeacd742|5.0       |18.0             |1                 |146.9      |1                 |
|398dee917bb02e71f1fd9653193b2396|5.0       |5.0              |1                 |97.87      |1                 |


In [20]:
# === BƯỚC 2: Tạo label churn từ RFM (KHÔNG lấy recency làm feature!) ===
print("Bước 2: Tạo label churn...")
rfm_with_churn = rfm_df.withColumn(
    "churn",
    F.when(F.col("recency") > 90, 1.0).otherwise(0.0)
)

# Chỉ lấy monetary, m_score — TUYỆT ĐỐI không lấy recency, r_score
ml_df = (
    rfm_with_churn
    .select("customer_unique_id", "frequency", "monetary", "f_score", "m_score", "churn")
    .join(customer_features, on="customer_unique_id", how="inner")
)

print(f"   → ml_df: {ml_df.count():,} dòng")
ml_df.show(5)

Bước 2: Tạo label churn...
   → ml_df: 93,358 dòng
+--------------------+---------+--------+-------+-------+-----+----------+-----------------+------------------+-----------+------------------+
|  customer_unique_id|frequency|monetary|f_score|m_score|churn|avg_review|avg_delivery_days|total_items_bought|avg_payment|category_diversity|
+--------------------+---------+--------+-------+-------+-----+----------+-----------------+------------------+-----------+------------------+
|aaeafdd832692cf3b...|        1|   92.74|      3|      3|  1.0|       5.0|             21.0|                 1|      92.74|                 1|
|7113a50db7ae2bc6f...|        1|   48.01|      3|      4|  1.0|       5.0|              8.0|                 1|      48.01|                 1|
|e4c5964975cdce805...|        1|   32.69|      1|      4|  0.0|       5.0|              9.0|                 1|      32.69|                 1|
|0242e42b2210c764b...|        1|   146.9|      4|      2|  1.0|       5.0|             18.0

## 4.5 Xử lý Null và Chuẩn bị Label

Trước khi đưa vào mô hình, cần:
1. Điền giá trị 0.0 cho các cột null (ví dụ: khách hàng chưa có review → avg_review = 0)
2. Cast label sang kiểu `double` (yêu cầu của Spark MLlib)

In [21]:
# Danh sách features (KHÔNG có recency, r_score!)
feature_cols = [
    "monetary", "m_score",
    "avg_review", "avg_delivery_days", "total_items_bought",
    "avg_payment", "category_diversity",
]

print(f"Features ({len(feature_cols)}):")
for f in feature_cols:
    print(f"   • {f}")

# Xử lý null
fill_values = {col: 0.0 for col in feature_cols}
ml_df = ml_df.fillna(fill_values)

# Tạo label column
ml_df = ml_df.withColumn("label", F.col("churn").cast("double"))
ml_df = ml_df.filter(F.col("label").isNotNull())

# Thống kê phân bố
data_count = ml_df.count()
churn_1 = ml_df.filter(F.col("label") == 1.0).count()
churn_0 = ml_df.filter(F.col("label") == 0.0).count()

print(f"\nPhân bố dữ liệu:")
print(f"   Tổng: {data_count:,} khách hàng")
print(f"   Churn=1 (rời bỏ):     {churn_1:,} ({churn_1/data_count*100:.1f}%)")
print(f"   Churn=0 (giữ lại):    {churn_0:,} ({churn_0/data_count*100:.1f}%)")

Features (7):
   • monetary
   • m_score
   • avg_review
   • avg_delivery_days
   • total_items_bought
   • avg_payment
   • category_diversity

Phân bố dữ liệu:
   Tổng: 93,358 khách hàng
   Churn=1 (rời bỏ):     74,838 (80.2%)
   Churn=0 (giữ lại):    18,520 (19.8%)


## 4.6 Xử lý mất cân bằng dữ liệu (Class Imbalance)

### Vấn đề:
Tỷ lệ Churn=1 chiếm ~80.2%, Churn=0 chiếm ~19.8%. Nếu không xử lý, mô hình sẽ thiên lệch về lớp đa số.

### Giải pháp: Class Weights (Trọng số lớp)
Thay vì dùng SMOTE (tạo mẫu giả), ta dùng **trọng số nghịch đảo tần suất** (inverse frequency weighting):

$$w_{class} = \frac{N_{total}}{2 \times N_{class}}$$

- Lớp thiểu số (ít mẫu hơn) → trọng số cao hơn → mô hình "chú ý" hơn
- Lớp đa số (nhiều mẫu hơn) → trọng số thấp hơn → tránh thiên lệch

> **Tại sao dùng Class Weights thay vì SMOTE?**
> - SMOTE tạo mẫu giả có thể gây nhiễu
> - Class Weights là built-in trong Spark MLlib, không cần thêm thư viện
> - Hiệu quả tương đương nhưng đơn giản hơn

In [22]:
# Tính class weights
w_1 = data_count / (2 * churn_1) if churn_1 > 0 else 1.0
w_0 = data_count / (2 * churn_0) if churn_0 > 0 else 1.0

print(f"Class Weights:")
print(f"   w(Churn=1): {w_1:.4f}")
print(f"   w(Churn=0): {w_0:.4f}")
print(f"   → Lớp thiểu số được tăng trọng số {w_0/w_1:.2f}x" if w_0 > w_1 
      else f"   → Lớp thiểu số được tăng trọng số {w_1/w_0:.2f}x")

# Thêm cột weight
ml_df = ml_df.withColumn(
    "weight", 
    F.when(F.col("label") == 1.0, w_1).otherwise(w_0)
)

Class Weights:
   w(Churn=1): 0.6237
   w(Churn=0): 2.5205
   → Lớp thiểu số được tăng trọng số 4.04x


## 4.7 Chia dữ liệu Train/Test

**Tỷ lệ**: 80% Train - 20% Test

**`seed=42`**: Đảm bảo kết quả lặp lại được (reproducible) - ai chạy lại cũng ra cùng kết quả.

In [23]:
# Chia train/test
train_df, test_df = ml_df.randomSplit([0.8, 0.2], seed=42)

train_count = train_df.count()
test_count = test_df.count()
print(f"Chia dữ liệu:")
print(f"   Train: {train_count:,} ({train_count/(train_count+test_count)*100:.0f}%)")
print(f"   Test:  {test_count:,} ({test_count/(train_count+test_count)*100:.0f}%)")

# Kiểm tra phân bố churn trong train/test
train_churn = train_df.filter(F.col("label")==1).count()
test_churn = test_df.filter(F.col("label")==1).count()
print(f"\n   Train Churn rate: {train_churn/train_count*100:.1f}%")
print(f"   Test  Churn rate: {test_churn/test_count*100:.1f}%")

Chia dữ liệu:
   Train: 74,861 (80%)
   Test:  18,497 (20%)

   Train Churn rate: 80.2%
   Test  Churn rate: 80.2%


## 4.8 Vector Assembler

Spark MLlib yêu cầu tất cả features phải được gộp thành **một cột vector duy nhất**. `VectorAssembler` làm nhiệm vụ này:

```
[monetary, m_score, avg_review, ...] → Vector(monetary, m_score, avg_review, ...)
```

`handleInvalid="skip"`: Bỏ qua các dòng có giá trị bất thường (NaN, Infinity).

In [24]:
# Tạo VectorAssembler
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features",
    handleInvalid="skip"
)
print(f"VectorAssembler: {len(feature_cols)} features -> 1 vector")

VectorAssembler: 7 features -> 1 vector


## 4.9 Mô hình 1: Random Forest Classifier

### Random Forest là gì?
Tập hợp nhiều cây quyết định (Decision Tree), mỗi cây học trên một phần dữ liệu ngẫu nhiên. Kết quả cuối cùng là **bỏ phiếu đa số** (majority voting).

### Hyperparameters:
| Tham số | Giá trị | Ý nghĩa |
|---------|---------|---------|
| `numTrees` | 50 | Số cây quyết định trong rừng |
| `maxDepth` | 6 | Độ sâu tối đa mỗi cây (tránh overfitting) |
| `weightCol` | "weight" | Cột trọng số để xử lý mất cân bằng |
| `seed` | 42 | Reproducibility |

### Tại sao chọn Random Forest?
- Xử lý tốt quan hệ phi tuyến giữa features
- Cho biết **tầm quan trọng** của từng feature
- Ít bị overfitting nhờ ensemble (kết hợp nhiều cây)
- Không cần feature scaling

In [25]:
# === RANDOM FOREST ===
print("Đang huấn luyện Random Forest Classifier...")
rf = RandomForestClassifier(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    numTrees=50,
    maxDepth=6,
    seed=42
)

# Tạo pipeline: Assembler → RandomForest
rf_pipeline = Pipeline(stages=[assembler, rf])
rf_model = rf_pipeline.fit(train_df)

# Dự đoán trên tập test
rf_preds = rf_model.transform(test_df)
print("Huấn luyện Random Forest hoàn tất!")

# Xem mẫu kết quả
rf_preds.select("customer_unique_id", "label", "prediction", "probability").show(5, truncate=False)

Đang huấn luyện Random Forest Classifier...
Huấn luyện Random Forest hoàn tất!
+--------------------------------+-----+----------+---------------------------------------+
|customer_unique_id              |label|prediction|probability                            |
+--------------------------------+-----+----------+---------------------------------------+
|00053a61a98854899e70ed204dd4bafe|1.0  |1.0       |[0.3322608704376251,0.6677391295623749]|
|000fbf0473c10fc1ab6f8d2d286ce20c|0.0  |0.0       |[0.5713710467392188,0.4286289532607813]|
|0011857aff0e5871ce5eb429f21cdaf5|1.0  |0.0       |[0.5947854080381663,0.4052145919618338]|
|001ae5a1788703d64536c30362503e49|1.0  |1.0       |[0.2808721566378183,0.7191278433621817]|
|0032c76b20340da25249092a268ce66c|1.0  |1.0       |[0.2824058777023523,0.7175941222976476]|
+--------------------------------+-----+----------+---------------------------------------+
only showing top 5 rows



## 4.10 Mô hình 2: Logistic Regression

### Logistic Regression là gì?
LR dùng hàm sigmoid để chuyển đổi tổ hợp tuyến tính của features thành xác suất (0-1):

$$P(churn=1) = \frac{1}{1 + e^{-(w_0 + w_1x_1 + w_2x_2 + ...)}}$$

### Hyperparameters:
| Tham số | Giá trị | Ý nghĩa |
|---------|---------|---------|
| `maxIter` | 100 | Số vòng lặp tối đa |
| `regParam` | 0.01 | Hệ số regularization (tránh overfitting) |

### Tại sao cần so sánh 2 mô hình?
- Kiểm chứng chéo (cross-validate) kết quả
- Chọn mô hình tốt nhất dựa trên metrics
- Hiểu đặc điểm dữ liệu (tuyến tính hay phi tuyến)

In [26]:
# === LOGISTIC REGRESSION ===
print("Đang huấn luyện Logistic Regression...")
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label",
    weightCol="weight",
    maxIter=100,
    regParam=0.01
)

lr_pipeline = Pipeline(stages=[assembler, lr])
lr_model = lr_pipeline.fit(train_df)

# Dự đoán trên tập test
lr_preds = lr_model.transform(test_df)
print("Huấn luyện Logistic Regression hoàn tất!")

Đang huấn luyện Logistic Regression...
Huấn luyện Logistic Regression hoàn tất!


## 4.11 Đánh giá và So sánh hai Mô hình

### Các chỉ số đánh giá:
| Chỉ số | Ý nghĩa | Quan trọng khi nào? |
|--------|---------|---------------------|
| **Accuracy** | Tỷ lệ dự đoán đúng tổng thể | Dữ liệu cân bằng |
| **Precision** | Trong số dự đoán Churn, bao nhiêu % đúng? | Tránh lãng phí marketing |
| **Recall** | Trong số khách thực sự Churn, bao nhiêu % phát hiện? | Không bỏ sót khách |
| **F1-Score** | Trung bình điều hòa Precision & Recall | Cân bằng cả hai |
| **AUC-ROC** | Khả năng phân biệt giữa 2 lớp (0.5=random, 1.0=perfect) | Đánh giá tổng thể |


In [27]:
# === ĐÁNH GIÁ CẢ HAI MÔ HÌNH ===

# Tạo các evaluator
auc_eval = BinaryClassificationEvaluator(labelCol="label", metricName="areaUnderROC")
acc_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="accuracy")
f1_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="f1")
prec_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedPrecision")
rec_eval = MulticlassClassificationEvaluator(labelCol="label", metricName="weightedRecall")

# Random Forest metrics
rf_metrics = {
    "AUC-ROC":   round(auc_eval.evaluate(rf_preds), 4),
    "Accuracy":  round(acc_eval.evaluate(rf_preds), 4),
    "F1-Score":  round(f1_eval.evaluate(rf_preds), 4),
    "Precision": round(prec_eval.evaluate(rf_preds), 4),
    "Recall":    round(rec_eval.evaluate(rf_preds), 4),
}

# Logistic Regression metrics
lr_metrics = {
    "AUC-ROC":   round(auc_eval.evaluate(lr_preds), 4),
    "Accuracy":  round(acc_eval.evaluate(lr_preds), 4),
    "F1-Score":  round(f1_eval.evaluate(lr_preds), 4),
    "Precision": round(prec_eval.evaluate(lr_preds), 4),
    "Recall":    round(rec_eval.evaluate(lr_preds), 4),
}

# Hiển thị so sánh
print("=" * 65)
print("  SO SÁNH HAI MÔ HÌNH (KHÔNG dùng recency - tránh Data Leakage)")
print("=" * 65)
print(f"{'Metric':<15} {'Random Forest':>15} {'Logistic Regression':>20}")
print("-" * 55)
for metric in ["AUC-ROC", "Accuracy", "F1-Score", "Precision", "Recall"]:
    print(f"  {metric:<13} {rf_metrics[metric]:>13.4f} {lr_metrics[metric]:>18.4f}")

# Chọn model tốt nhất theo AUC
if rf_metrics["AUC-ROC"] >= lr_metrics["AUC-ROC"]:
    best_name = "Random Forest"
    best_preds = rf_preds
    best_model = rf_model
else:
    best_name = "Logistic Regression"
    best_preds = lr_preds
    best_model = lr_model

print(f"\nBest model: {best_name} (AUC = {max(rf_metrics['AUC-ROC'], lr_metrics['AUC-ROC'])})")

  SO SÁNH HAI MÔ HÌNH (KHÔNG dùng recency - tránh Data Leakage)
Metric            Random Forest  Logistic Regression
-------------------------------------------------------
  AUC-ROC              0.6892             0.6882
  Accuracy             0.5786             0.5784
  F1-Score             0.6198             0.6196
  Precision            0.7787             0.7790
  Recall               0.5786             0.5784

Best model: Random Forest (AUC = 0.6892)


## 4.12 Ma trận nhầm lẫn (Confusion Matrix)

### Ma trận:
```
                      Dự đoán: Không rời bỏ    Dự đoán: Rời bỏ
Thực tế: Không rời bỏ     TN (Đúng)            FP (Sai - Type I)
Thực tế: Rời bỏ           FN (Sai - Type II)   TP (Đúng)
```

- **TN (True Negative)**: Dự đoán ĐÚNG khách không rời bỏ
- **TP (True Positive)**: Dự đoán ĐÚNG khách rời bỏ
- **FP (False Positive)**: Dự đoán khách rời bỏ nhưng thực tế không → Lãng phí chi phí marketing
- **FN (False Negative)**: Bỏ sót khách thực sự rời bỏ → Mất khách

In [28]:
# === CONFUSION MATRIX ===
# Dùng filter + count thay vì collect (an toàn hơn với dữ liệu lớn)
tp = best_preds.filter((F.col("label") == 1.0) & (F.col("prediction") == 1.0)).count()
tn = best_preds.filter((F.col("label") == 0.0) & (F.col("prediction") == 0.0)).count()
fp = best_preds.filter((F.col("label") == 0.0) & (F.col("prediction") == 1.0)).count()
fn = best_preds.filter((F.col("label") == 1.0) & (F.col("prediction") == 0.0)).count()

print(f"MA TRẬN NHẦM LẪN - {best_name.upper()}")
print("=" * 50)
print(f"                    Dự đoán=0    Dự đoán=1")
print(f"   Thực tế=0 (Giữ)   TN={tn:>6,}    FP={fp:>6,}")
print(f"   Thực tế=1 (Churn)  FN={fn:>6,}    TP={tp:>6,}")
print()
print(f"   Dự đoán đúng:  {tp+tn:,} ({(tp+tn)/(tp+tn+fp+fn)*100:.1f}%)")
print(f"   Dự đoán sai:   {fp+fn:,} ({(fp+fn)/(tp+tn+fp+fn)*100:.1f}%)")
print(f"   Precision (Churn): {tp/(tp+fp)*100:.1f}%" if (tp+fp) > 0 else "")
print(f"   Recall (Churn):    {tp/(tp+fn)*100:.1f}%" if (tp+fn) > 0 else "")

MA TRẬN NHẦM LẪN - RANDOM FOREST
                    Dự đoán=0    Dự đoán=1
   Thực tế=0 (Giữ)   TN= 2,794    FP=   876
   Thực tế=1 (Churn)  FN= 6,918    TP= 7,909

   Dự đoán đúng:  10,703 (57.9%)
   Dự đoán sai:   7,794 (42.1%)
   Precision (Churn): 90.0%
   Recall (Churn):    53.3%


## 4.13 Tầm quan trọng đặc trưng (Feature Importance)

### Tại sao cần biết?
Feature importance cho biết yếu tố nào ảnh hưởng **NHIỀU NHẤT** đến việc khách hàng rời bỏ. Điều này giúp doanh nghiệp:
- Tập trung nguồn lực vào đúng chỗ
- Hiểu hành vi khách hàng
- Cải thiện dịch vụ ở điểm then chốt


In [29]:
# === FEATURE IMPORTANCE (từ Random Forest) ===
rf_classifier = rf_model.stages[-1]  # RandomForestClassificationModel
importances = rf_classifier.featureImportances.toArray().tolist()

# Tạo dict và sắp xếp giảm dần
fi_dict = {}
for i, name in enumerate(feature_cols):
    fi_dict[name] = round(importances[i], 4) if i < len(importances) else 0.0

fi_sorted = dict(sorted(fi_dict.items(), key=lambda x: x[1], reverse=True))

print("TẦM QUAN TRỌNG ĐẶC TRƯNG (Random Forest)")
print("=" * 55)
for feat, imp in fi_sorted.items():
    bar = "█" * int(imp * 80)
    print(f"  {feat:25s} {imp:.4f}  {bar}")

print()
top_feature = list(fi_sorted.keys())[0]
print(f"Feature quan trọng nhất: {top_feature}")

TẦM QUAN TRỌNG ĐẶC TRƯNG (Random Forest)
  avg_delivery_days         0.9185  █████████████████████████████████████████████████████████████████████████
  avg_review                0.0326  ██
  monetary                  0.0176  █
  avg_payment               0.0172  █
  total_items_bought        0.0063  
  category_diversity        0.0045  
  m_score                   0.0033  

Feature quan trọng nhất: avg_delivery_days


## 4.14 Lưu Mô hình và Kết quả Dự đoán

### Lưu gì?
1. **Mô hình** → HDFS (để có thể load lại sau)
2. **Churn predictions** → HDFS Gold layer (Parquet) → Sau đó export sang MongoDB

### Trích xuất Churn Probability
Spark MLlib trả về cột `probability` dạng Vector `[P(class=0), P(class=1)]`.
Ta dùng `vector_to_array()` để chuyển thành array, rồi lấy phần tử thứ 2 là `P(churn=1)`.

In [30]:
# === LƯU MÔ HÌNH ===
HDFS_MODELS = "hdfs://localhost:9000/user/bigdata/olist/models"
model_path = f"{HDFS_MODELS}/churn_classifier"
best_model.write().overwrite().save(model_path)
print(f"Đã lưu model tại: {model_path}")

# === LƯU CHURN PREDICTIONS ===
# Trích xuất churn probability từ vector
churn_output = (
    best_preds
    .withColumn("prob_array", vector_to_array("probability"))
    .withColumn("churn_probability", F.round(F.element_at("prob_array", 2), 4))
    .select("customer_unique_id", "label", "prediction", "churn_probability")
)

output_path = f"{HDFS_GOLD}/churn_predictions"
churn_output.coalesce(2).write.mode("overwrite").parquet(output_path)
print(f"Đã lưu predictions tại: {output_path}")

# Xem mẫu kết quả
print("\nMẫu kết quả dự đoán:")
churn_output.orderBy(F.col("churn_probability").desc()).show(10, truncate=False)

Đã lưu model tại: hdfs://localhost:9000/user/bigdata/olist/models/churn_classifier
Đã lưu predictions tại: hdfs://localhost:9000/user/bigdata/olist/gold/churn_predictions

Mẫu kết quả dự đoán:
+--------------------------------+-----+----------+-----------------+
|customer_unique_id              |label|prediction|churn_probability|
+--------------------------------+-----+----------+-----------------+
|ce0640b5ebf6dd6badf3f245d2cd60d0|1.0  |1.0       |0.8167           |
|0d205c3a77c323684c7e9fa525448110|1.0  |1.0       |0.8087           |
|06c41bfdcfbb58cc7dfab0a107cd6436|1.0  |1.0       |0.8087           |
|6dfd53d0b64aa10fd66b8a7a4ad4461c|1.0  |1.0       |0.8087           |
|0bc6a141b17a9e42140deaa6e5cdb9df|1.0  |1.0       |0.8087           |
|decf5e95920bb2dc3fe659c102dbbf70|1.0  |1.0       |0.8087           |
|5376bd8df26ab646ce3e4c874bc693b9|1.0  |1.0       |0.8087           |
|dfc8c1b8f452a51070d587e7faec476d|1.0  |1.0       |0.8087           |
|f4e2749a803a0a9019e6c69bc0e1b2a5|1.0

In [31]:
# Dừng SparkSession
spark.stop()
print("SparkSession đã dừng.")

SparkSession đã dừng.
